Import Statements

In [148]:
import pandas as pd
from datetime import datetime,timedelta
from typing import Dict, List, Union, Final
import numpy as np

Configuration and Inputs

In [149]:

START_DATE: Final[str] = datetime(2026,1,1).date()
lot_size=550

temp=pd.read_csv('Feb_FUT.csv')
Fut=temp[['Date','Settlement Price']].copy()

Making the date into indexes for the settlement prices

In [150]:
if 'Date' in Fut.columns:
 Fut['Date'] = pd.to_datetime(Fut['Date'], dayfirst=True)
 Fut['Settlement Price'] = pd.to_numeric(Fut['Settlement Price'].astype(str).str.replace(',', ''),errors='coerce')
 Fut.set_index('Date', inplace=True)
 Fut.sort_index(inplace=True)
 Fut=Fut.loc['2026-01']

Creating Accounts Dataframe

In [154]:
Accounts =pd.DataFrame(columns=["Date", "Margin_A/C", "Borrowing_A/C", "Interest_Due"])
if 'Date' in Accounts.columns:
 Accounts['Date'] = pd.to_datetime(Accounts['Date'], dayfirst=True)
 Accounts.set_index('Date', inplace=True)




F0=Fut.loc['2026-01-01', 'Settlement Price']

Contract_Price=F0*lot_size
Num_Contracts=(round(15000000/Contract_Price))

Money_used=0.3*(Contract_Price)*Num_Contracts

Accounts.loc[START_DATE]=(Money_used/100000,0.0,0.0)

Money_left=5000000.0-Money_used
Margin=Money_used*(2/3)

Borrowed=0.0
Interest=0.0
rate=0.0985/365

In [155]:

P0=F0
skip_date = pd.Timestamp('2026-01-01').date()
for i in Fut.index:
    clean=i.date()
    if clean==skip_date:
     continue
    
    P1=Fut.at[i, 'Settlement Price']
    Change = (P1-P0)*(Num_Contracts)*lot_size
    Money_used+=Change
    Interest+=Borrowed*rate


      

    P0=P1
    Accounts.loc[clean]=(Money_used/100000,Borrowed/100000,Interest/100000)
       
    if(Money_left<Margin):
      Money_used=Margin
      Money_left+=Change
      if(Money_left<0):
       Borrowed= -(Money_left)


Accounts.to_excel('FEB_FUT.xlsx')
      